# Demo 02. Newton's Form and the Divided-Difference Table

**Standard 1** (interpolation construction), Week 1.

The interpolating polynomial through a set of points is unique but admits several representations. Demo 01 used the Lagrange form. This demo constructs the same polynomial in Newton's form, whose coefficients come from a divided-difference table and for which adding a data point costs a single additional term.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
# Enable interactive ipywidgets sliders. try/except so the notebook also runs
# in plain Jupyter (outside Colab) without error.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown, Checkbox

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Divided differences

$$ p_n(x) = \sum_{k=0}^{n} f[x_0,\dots,x_k]\prod_{j=0}^{k-1}(x - x_j), \qquad f[x_i,\dots,x_{i+j}] = \frac{f[x_{i+1},\dots,x_{i+j}] - f[x_i,\dots,x_{i+j-1}]}{x_{i+j}-x_i}. $$

The coefficients $f[x_0,\dots,x_k]$ occupy the top row of the triangular table. Adding a node extends the table by one anti-diagonal and leaves the existing coefficients unchanged, which makes Newton's form suited to data that arrive incrementally.

In [ ]:
# ---------------------------------------------------------------------------
# The divided-difference table
# ---------------------------------------------------------------------------
# Newton's form writes the interpolating polynomial as
#
#   p(x) = c0 + c1 (x-x0) + c2 (x-x0)(x-x1) + ... + cn (x-x0)...(x-x_{n-1})
#
# where the coefficients c_k are "divided differences" f[x0,...,xk]. They are
# built from a triangular table, each entry a difference of two neighbours in
# the previous column, divided by the spread of the x's involved.

def divided_differences(x, y):
    """Build the divided-difference table for data (x[i], y[i]).

    Parameters
    ----------
    x : array (n,)  distinct nodes
    y : array (n,)  values, y[i] = f(x[i])

    Returns
    -------
    coeffs : array (n,)   Newton coefficients c0..c_{n-1} (the top row of the table)
    table  : array (n,n)  full lower-triangular table (zeros above the diagonal used)

    The recurrence:  table[i, j] = ( table[i+1, j-1] - table[i, j-1] )
                                   / ( x[i+j] - x[i] )
    Column 0 is just the y-values; each later column is one order higher.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    table = np.zeros((n, n))
    table[:, 0] = y                          # 0th divided differences are the y's
    for j in range(1, n):                    # column = order of the difference
        for i in range(n - j):               # row
            table[i, j] = (table[i + 1, j - 1] - table[i, j - 1]) / (x[i + j] - x[i])
    coeffs = table[0, :].copy()              # Newton coefficients live on the top row
    return coeffs, table


def newton_eval(x_nodes, coeffs, x):
    """Evaluate the Newton-form polynomial with the given coefficients at x.

    Uses a nested (Horner-like) scheme, which is both fast and numerically
    steadier than expanding the products explicitly:

        p = c_{n-1}
        p = p*(x - x_{n-2}) + c_{n-2}
        ...
        p = p*(x - x_0) + c_0
    """
    x = np.asarray(x, dtype=float)
    n = len(coeffs)
    result = np.full_like(x, coeffs[-1])     # start from the highest coefficient
    for k in range(n - 2, -1, -1):           # fold in the rest, high to low
        result = result * (x - x_nodes[k]) + coeffs[k]
    return result

In [ ]:
# ---------------------------------------------------------------------------
# A readable printout of the triangular table
# ---------------------------------------------------------------------------
def print_dd_table(x, table):
    """Pretty-print the divided-difference table as a triangle.

    Row i, column j holds f[x_i, ..., x_{i+j}]. The Newton coefficients are the
    top row (j-th coefficient = table[0, j]).
    """
    n = len(x)
    print("  x_i   |  f[.] (order increases left -> right)")
    print("-" * 60)
    for i in range(n):
        row = f"{x[i]:6.3f} |"
        for j in range(n - i):
            row += f" {table[i, j]:10.4f}"
        print(row)

In [ ]:
# ---------------------------------------------------------------------------
# Incremental construction
# ---------------------------------------------------------------------------
# Add one more data point and only one new coefficient appears at the end --
# all previous coefficients are unchanged. (Lagrange, by contrast, must rebuild
# every basis polynomial from scratch.) Watch the coefficient list grow by
# exactly one entry each time you raise the node count.

def f_demo(x):
    """Function to sample. A smooth oscillatory function."""
    return np.sin(2 * x) + 0.5 * x

def show_newton(n_nodes=5, show_table=True, a=0.0, b=5.0):
    """Sample f_demo at n_nodes equally spaced points, build the Newton
    interpolant, plot it against f, and optionally print the table + coeffs."""
    x = np.linspace(a, b, n_nodes)           # nodes
    y = f_demo(x)                            # data values
    coeffs, table = divided_differences(x, y)

    xx = np.linspace(a, b, 400)
    p = newton_eval(x, coeffs, xx)

    plt.figure()
    plt.plot(xx, f_demo(xx), "k-", lw=2, label="f(x)")
    plt.plot(xx, p, "r--", lw=2, label=f"Newton interpolant (n={n_nodes})")
    plt.plot(x, y, "bo", ms=7, label="data")
    plt.legend(); plt.xlabel("x"); plt.title("Newton-form interpolation")
    plt.show()

    if show_table:
        print_dd_table(x, table)
        print("\nNewton coefficients c0..c{}:".format(n_nodes - 1))
        print(np.array2string(coeffs, precision=4))
        print("\nNote: raise n by one and the earlier coefficients stay put, "
              "only a new last term is added.")

show_newton(5)

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: slide the number of nodes; toggle the table
# ---------------------------------------------------------------------------
interact(
    show_newton,
    n_nodes=IntSlider(min=2, max=12, step=1, value=5, description="# nodes"),
    show_table=Checkbox(value=True, description="show table"),
    a=(-1.0, 2.0, 0.5), b=(3.0, 8.0, 0.5),
);

## Summary

- Newton and Lagrange produce the same unique interpolating polynomial in different representations.
- Divided differences are computed in $O(n^2)$; evaluation by nested multiplication is $O(n)$.
- Adding one node adds exactly one coefficient and reuses prior work. The coefficient list grows by one entry as the node count increases.